<a href="https://colab.research.google.com/github/fareyahossain835-bit/flyrank-ml/blob/main/Capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Predicting Search Performance Decline for Content Review Prioritization

### A Leakage-Aware Machine Learning Study Using the FlyRank Search Performance Warehouse

**Author:** Fareya Hossain  
**Track:** Machine Learning  
**Project:** Google Search Ranking & Discoverability Capstone  
**Dataset:** FlyRank Internship Warehouse

---

## Research Question

Can historical search-performance signals be used to identify content items that are more likely to experience a significant decline in search impressions in a future observation window?

## Decision Supported

The goal is not to predict Google's ranking algorithm or prove why a page loses visibility.

Instead, this project builds a decision-support system that ranks content items by observed future-decline risk so that a limited review team can prioritize which items deserve attention first.

## Approach

We construct features from a historical observation window, define decline using a separate future window, compare a transparent baseline against Logistic Regression and Random Forest models, and evaluate the models using time-aware validation and review-queue metrics such as Precision@K.

The final output is a ranked review queue with interpretable reason codes.


## Public-Safety and Research Framing

This analysis uses pseudonymized FlyRank search-performance data.

The notebook will not expose:

- client names
- domains
- URLs
- raw search queries
- page titles
- credentials
- private identifiers

The analysis is observational. Model predictions represent historical patterns in the available data and should be interpreted as decision-support signals rather than guarantees.

This project does not claim to identify Google's ranking factors, Google's algorithm, or causal effects of content changes.


In [ ]:
!pip -q install duckdb pandas numpy scikit-learn matplotlib seaborn pyarrow huggingface_hub

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)

from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

print("Environment ready.")

## 1. Data Configuration

The FlyRank warehouse is stored as Parquet data and is large enough that loading the complete daily table into pandas is unnecessary.

DuckDB is used to aggregate the required observations inside the database engine. Only the resulting feature table is transferred to pandas.

This follows the recommended workflow for the warehouse: aggregate per content item in DuckDB and then model the smaller result using pandas and scikit-learn.

In [ ]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN not found in Colab Secrets.")

login(token=HF_TOKEN)

print("Hugging Face authentication successful.")

In [ ]:
con = duckdb.connect()

con.execute("""
INSTALL httpfs;
LOAD httpfs;
""")

print("DuckDB version:", duckdb.__version__)

In [ ]:
import os
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN not found in Colab Secrets.")

os.environ["HF_TOKEN"] = HF_TOKEN

print("HF token loaded successfully.")

In [ ]:
from huggingface_hub import login

login(token=HF_TOKEN)

print("Hugging Face login successful.")

In [ ]:
con = duckdb.connect()

con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
);
""")

print("DuckDB Hugging Face authentication configured.")

In [ ]:
HF_DATASET = "hf://datasets/FlyRank/internship-warehouse"

print("Dataset:", HF_DATASET)


In [ ]:
HF_DATASET = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients": f"{HF_DATASET}/dim_clients.parquet",
    "dim_content": f"{HF_DATASET}/dim_content.parquet",
    "fact_query_90d": f"{HF_DATASET}/fact_content_query_90d.parquet",

    # IMPORTANT: recursive path because daily data is partitioned by month
    "fact_daily": f"{HF_DATASET}/fact_content_daily_performance/**/*.parquet"
}

TABLES

In [ ]:
for name, path in TABLES.items():
    print(f"\n{name}")
    print(path)

In [ ]:
for name, path in TABLES.items():
    print(f"\n{name}")
    print(path)

In [ ]:
query_schema = con.sql(f"""
DESCRIBE SELECT *
FROM read_parquet('{TABLES["fact_query_90d"]}')
""").df()

display(query_schema)

## 2. Data Contract

The primary modeling grain will be:

**one row = one client-content item**

The daily performance table provides observations over time.

The query-level 90-day table provides aggregated signals describing how impressions are distributed across queries without exposing the raw query text.

The modeling design intentionally avoids raw queries, URLs, titles, domains, client names, or other identifying fields.


In [ ]:
date_range = con.sql(f"""
SELECT
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date,
    COUNT(*) AS row_count
FROM read_parquet(
    '{TABLES["fact_daily"]}',
    hive_partitioning = true
)
""").df()

display(date_range)

In [ ]:
daily_schema = con.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet(
    '{TABLES["fact_daily"]}',
    hive_partitioning = true
)
""").df()

display(daily_schema)

In [ ]:
query_schema = con.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet(
    '{TABLES["fact_query_90d"]}'
)
""").df()

display(query_schema)

## 3. Research Design

This project studies whether historical search-performance patterns can help prioritize content items that experience a substantial decline in impressions during a future observation window.

The modeling design separates historical features from the future outcome.

For each prediction cutoff:

- Historical window: previous 90 days
- Recent comparison window: previous 30 days
- Future outcome window: next 30 days

The model only receives information available on or before the cutoff date.

The target is calculated exclusively from the future window.

This time-aware design reduces the risk of target leakage and better represents the real-world decision of prioritizing content for future review.'

In [ ]:
MIN_DATE = pd.to_datetime("2025-01-27")
MAX_DATE = pd.to_datetime("2026-06-30")

FEATURE_DAYS = 90
TARGET_DAYS = 30

# We need:
# 90 days before cutoff for features
# 30 days after cutoff for target

FIRST_CUTOFF = MIN_DATE + pd.Timedelta(days=FEATURE_DAYS)
LAST_CUTOFF = MAX_DATE - pd.Timedelta(days=TARGET_DAYS)

cutoffs = pd.date_range(
    start=FIRST_CUTOFF,
    end=LAST_CUTOFF,
    freq="30D"
)

print("First cutoff:", FIRST_CUTOFF.date())
print("Last cutoff:", LAST_CUTOFF.date())
print("Number of cutoffs:", len(cutoffs))
print("\nCutoffs:")
print([d.date() for d in cutoffs])

## 4. Historical Feature Construction

The daily performance table is used as the primary source for leakage-safe historical features.

For each client-content pair and prediction cutoff, we calculate:

- 90-day impressions
- 90-day clicks
- 90-day CTR
- 90-day average position
- 30-day impressions
- 30-day clicks
- 30-day CTR
- recent-to-historical impression ratio

Only observations occurring on or before the cutoff are included in these features.


In [ ]:
con.execute(f"""
CREATE OR REPLACE TABLE monthly_content AS
SELECT
    DATE_TRUNC('month', report_date)::DATE AS month,
    client_hash_id,
    content_hash_id,

    SUM(COALESCE(gsc_impressions, 0)) AS impressions,
    SUM(COALESCE(gsc_clicks, 0)) AS clicks,
    AVG(gsc_avg_position) AS avg_position,

    SUM(COALESCE(ga4_pageviews, 0)) AS pageviews,
    SUM(COALESCE(ga4_sessions, 0)) AS sessions,
    SUM(COALESCE(sessions_organic, 0)) AS sessions_organic,
    SUM(COALESCE(sessions_ai, 0)) AS sessions_ai

FROM read_parquet(
    '{TABLES["fact_daily"]}',
    hive_partitioning = true
)

GROUP BY
    1, 2, 3
""")

print("Monthly aggregation completed.")

In [ ]:
# Check the compact dataset

monthly_check = con.sql("""
SELECT
    COUNT(*) AS row_count,
    MIN(month) AS min_month,
    MAX(month) AS max_month
FROM monthly_content
""").df()

display(monthly_check)

In [ ]:
# compact dataset

con.execute("""
COPY monthly_content
TO 'monthly_content.parquet'
(FORMAT PARQUET, COMPRESSION ZSTD)
""")

print("Saved successfully: monthly_content.parquet")

In [ ]:
monthly_sample = con.sql("""
SELECT *
FROM monthly_content
ORDER BY month
LIMIT 10
""").df()

display(monthly_sample)

## 5. Monthly Historical Features and Future Outcome

To make the analysis computationally efficient, the daily performance data was aggregated to a monthly client-content level.

For each client-content pair and prediction month, historical features are calculated using only months before the prediction month.

The future outcome is calculated from the following month.

A content item is labeled as declining when its next-month impressions are at least 20% lower than its previous-month impressions.

The prediction therefore uses historical information to prioritize content for future review.


In [ ]:
monthly_model_sql = """
WITH base AS (

    SELECT
        month,
        client_hash_id,
        content_hash_id,

        impressions,
        clicks,
        avg_position,
        pageviews,
        sessions,
        sessions_organic,
        sessions_ai,

        CASE
            WHEN impressions > 0
            THEN clicks::DOUBLE / impressions
            ELSE 0
        END AS ctr

    FROM monthly_content

    WHERE impressions >= 10
),

with_lag AS (

    SELECT
        *,

        LAG(impressions, 1) OVER (
            PARTITION BY client_hash_id, content_hash_id
            ORDER BY month
        ) AS prev_month_impressions,

        LAG(clicks, 1) OVER (
            PARTITION BY client_hash_id, content_hash_id
            ORDER BY month
        ) AS prev_month_clicks,

        LAG(avg_position, 1) OVER (
            PARTITION BY client_hash_id, content_hash_id
            ORDER BY month
        ) AS prev_month_position,

        LAG(ctr, 1) OVER (
            PARTITION BY client_hash_id, content_hash_id
            ORDER BY month
        ) AS prev_month_ctr

    FROM base
)

SELECT *
FROM with_lag
WHERE prev_month_impressions IS NOT NULL
"""

monthly_model = con.sql(
    monthly_model_sql
).df()

print(
    "Model rows:",
    f"{len(monthly_model):,}"
)

display(monthly_model.head())

In [ ]:
monthly_model = monthly_model.sort_values(
    [
        "client_hash_id",
        "content_hash_id",
        "month"
    ]
).reset_index(drop=True)

monthly_model["future_impressions"] = (
    monthly_model
    .groupby(
        [
            "client_hash_id",
            "content_hash_id"
        ]
    )["impressions"]
    .shift(-1)
)

monthly_model["future_change"] = (
    monthly_model["future_impressions"]
    -
    monthly_model["impressions"]
) / monthly_model["impressions"].replace(
    0,
    np.nan
)

monthly_model["is_declining"] = (
    monthly_model["future_change"] <= -0.20
).astype(int)

# The last month for each content has no future observation.
monthly_model = monthly_model[
    monthly_model["future_impressions"].notna()
].copy()

print(
    "Final modeling rows:",
    f"{len(monthly_model):,}"
)

print(
    "Future decline rate:",
    f"{monthly_model['is_declining'].mean():.3%}"
)

## 6. Leakage Control

The prediction is made at the end of each month.

All model features describe the current or historical state available at that prediction point.

The target uses only the following month's impressions.

Therefore, the future impression value used to construct the label is not included in the feature matrix.

A separate temporal train/test split is also used so that later months are not used to train the model evaluated on earlier months.


In [ ]:
FEATURES = [
    "impressions",
    "clicks",
    "ctr",
    "avg_position",
    "pageviews",
    "sessions",
    "sessions_organic",
    "sessions_ai",

    "prev_month_impressions",
    "prev_month_clicks",
    "prev_month_position",
    "prev_month_ctr"
]

TARGET = "is_declining"

UNSAFE_COLUMNS = [
    "future_impressions",
    "future_change",
    "is_declining"
]

print("Features:")
for feature in FEATURES:
    print(" -", feature)

In [ ]:
# Explicit leakage checks

assert "future_impressions" not in FEATURES
assert "future_change" not in FEATURES
assert TARGET not in FEATURES

for feature in FEATURES:
    assert "future" not in feature.lower()
    assert "target" not in feature.lower()
    assert "label" not in feature.lower()
    assert "declin" not in feature.lower()

print(" LEAKAGE TEST PASSED")
print(" Future target variables are excluded from features.")

In [ ]:
available_months = sorted(
    monthly_model["month"].unique()
)

split_index = int(
    len(available_months) * 0.80
)

train_months = available_months[:split_index]
test_months = available_months[split_index:]

train = monthly_model[
    monthly_model["month"].isin(train_months)
].copy()

test = monthly_model[
    monthly_model["month"].isin(test_months)
].copy()

print("TRAIN PERIOD")
print(
    train["month"].min(),
    "→",
    train["month"].max()
)

print("\nTEST PERIOD")
print(
    test["month"].min(),
    "→",
    test["month"].max()
)

print("\nTrain rows:", f"{len(train):,}")
print("Test rows:", f"{len(test):,}")

In [ ]:
assert (
    train["month"].max()
    <
    test["month"].min()
)

print(" TIME SPLIT PASSED")

## 7. Baseline

A simple rule-based baseline is used before machine learning.

The baseline predicts a future decline when the current month's impressions are already at least 20% lower than the previous month's impressions.

This provides a transparent benchmark for evaluating whether the machine-learning models provide additional predictive value.


In [ ]:
test["baseline_score"] = (
    1 -
    (
        test["impressions"] /
        test["prev_month_impressions"].replace(0, np.nan)
    )
)

test["baseline_score"] = test["baseline_score"].fillna(0)

test["baseline_pred"] = (
    test["baseline_score"] >= 0.20
).astype(int)

print("Baseline results:")
print(
    classification_report(
        test["is_declining"],
        test["baseline_pred"],
        digits=3,
        zero_division=0
    )
)

In [ ]:
X_train = train[FEATURES].copy()
y_train = train[TARGET].copy()

X_test = test[FEATURES].copy()
y_test = test[TARGET].copy()

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

In [ ]:
logistic_model = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        )
    )
])

logistic_model.fit(
    X_train,
    y_train
)

test["logistic_prob"] = (
    logistic_model.predict_proba(X_test)[:, 1]
)

test["logistic_pred"] = (
    test["logistic_prob"] >= 0.50
).astype(int)

print("Logistic Regression trained")

In [ ]:
 #Random Forest

rf_model = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "model",
        RandomForestClassifier(
            n_estimators=60,
            max_depth=8,
            min_samples_leaf=30,
            max_features="sqrt",
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        )
    )
])

rf_model.fit(
    X_train,
    y_train
)

test["rf_prob"] = (
    rf_model.predict_proba(X_test)[:, 1]
)

test["rf_pred"] = (
    test["rf_prob"] >= 0.50
).astype(int)

print(" Random Forest trained")

In [ ]:
def precision_at_k(y_true, scores, k=20):
    temp = pd.DataFrame({
        "actual": np.asarray(y_true),
        "score": np.asarray(scores)
    })

    top_k = (
        temp
        .sort_values("score", ascending=False)
        .head(k)
    )

    return top_k["actual"].mean()


baseline_p20 = precision_at_k(
    y_test,
    test["baseline_score"],
    20
)

logistic_p20 = precision_at_k(
    y_test,
    test["logistic_prob"],
    20
)

rf_p20 = precision_at_k(
    y_test,
    test["rf_prob"],
    20
)

print(f"Baseline Precision@20:          {baseline_p20:.3f}")
print(f"Logistic Regression Precision@20: {logistic_p20:.3f}")
print(f"Random Forest Precision@20:       {rf_p20:.3f}")

In [ ]:
results = []

models_to_evaluate = [
    (
        "Baseline",
        test["baseline_pred"],
        test["baseline_score"]
    ),
    (
        "Logistic Regression",
        test["logistic_pred"],
        test["logistic_prob"]
    ),
    (
        "Random Forest",
        test["rf_pred"],
        test["rf_prob"]
    )
]

for name, predictions, probabilities in models_to_evaluate:

    results.append({
        "model": name,

        "accuracy": accuracy_score(
            y_test,
            predictions
        ),

        "precision": precision_score(
            y_test,
            predictions,
            zero_division=0
        ),

        "recall": recall_score(
            y_test,
            predictions,
            zero_division=0
        ),

        "f1": f1_score(
            y_test,
            predictions,
            zero_division=0
        ),

        "roc_auc": roc_auc_score(
            y_test,
            probabilities
        ),

        "average_precision": average_precision_score(
            y_test,
            probabilities
        ),

        "precision_at_20": precision_at_k(
            y_test,
            probabilities,
            20
        )
    })

results_df = pd.DataFrame(results)

display(
    results_df.round(4)
)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 5))

plt.bar(
    results_df["model"],
    results_df["precision_at_20"]
)

plt.xlabel("Model")
plt.ylabel("Precision@20")
plt.title("Precision@20 Comparison")

plt.xticks(rotation=15)
plt.ylim(0, 1)

plt.tight_layout()

plt.savefig(
    "precision_at_20_comparison.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()

In [ ]:
rf_estimator = rf_model.named_steps["model"]

importance = pd.DataFrame({
    "feature": FEATURES,
    "importance": rf_estimator.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

display(importance)

In [ ]:
plot_data = importance.sort_values(
    "importance"
)

plt.figure(figsize=(10, 7))

plt.barh(
    plot_data["feature"],
    plot_data["importance"]
)

plt.xlabel("Feature Importance")
plt.ylabel("Feature")
plt.title("Random Forest Feature Importance")

plt.tight_layout()

plt.savefig(
    "feature_importance.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()

In [ ]:
recommendations = test.copy()

recommendations["risk_score"] = recommendations["rf_prob"]

recommendations = (
    recommendations
    .sort_values(
        "risk_score",
        ascending=False
    )
    .reset_index(drop=True)
)

recommendations["rank"] = (
    np.arange(len(recommendations)) + 1
)

recommendations["risk_tier"] = pd.cut(
    recommendations["risk_score"],
    bins=[-np.inf, 0.50, 0.75, np.inf],
    labels=["Low", "Medium", "High"]
)

display(
    recommendations[
        [
            "rank",
            "client_hash_id",
            "content_hash_id",
            "month",
            "risk_score",
            "risk_tier",
            "impressions",
            "prev_month_impressions"
        ]
    ].head(20)
)

In [ ]:
def get_reason(row):

    reasons = []

    if row["impressions"] < row["prev_month_impressions"]:
        reasons.append("Recent impression decline")

    if row["avg_position"] >= 10:
        reasons.append("Weak average position")

    if row["ctr"] < row["prev_month_ctr"]:
        reasons.append("CTR deterioration")

    if not reasons:
        reasons.append("Elevated predicted decline risk")

    return "; ".join(reasons[:2])


recommendations["reason_code"] = (
    recommendations.apply(
        get_reason,
        axis=1
    )
)


def get_action(tier):

    if tier == "High":
        return "Priority review"
    elif tier == "Medium":
        return "Content review"
    else:
        return "Monitor"


recommendations["recommended_action"] = (
    recommendations["risk_tier"]
    .apply(get_action)
)

display(
    recommendations.head(20)
)

In [ ]:
FINAL_COLUMNS = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "month",
    "risk_score",
    "risk_tier",
    "reason_code",
    "recommended_action",
    "impressions",
    "prev_month_impressions",
    "clicks",
    "ctr",
    "avg_position",
    "prev_month_position",
    "prev_month_ctr",
    "pageviews",
    "sessions",
    "sessions_organic",
    "sessions_ai"
]

final_queue = recommendations[FINAL_COLUMNS].copy()

final_queue.to_csv(
    "ranked_content_recommendations.csv",
    index=False
)

print("ranked_content_recommendations.csv created")

display(final_queue.head(20))

In [ ]:
top20 = recommendations.head(20)

actual_precision20 = (
    top20["is_declining"].mean()
)

print(
    f"Final Random Forest Precision@20: "
    f"{actual_precision20:.3f}"
)

print(
    f"Actual declining items in top 20: "
    f"{top20['is_declining'].sum()} / 20"
)

In [ ]:
import os
import shutil

os.makedirs("outputs/charts", exist_ok=True)

final_queue.to_csv(
    "outputs/ranked_content_recommendations.csv",
    index=False
)

results_df.to_csv(
    "outputs/model_results.csv",
    index=False
)

importance.to_csv(
    "outputs/feature_importance.csv",
    index=False
)

for filename in [
    "precision_at_20_comparison.png",
    "feature_importance.png"
]:
    if os.path.exists(filename):
        shutil.copy(
            filename,
            f"outputs/charts/{filename}"
        )

print("All capstone outputs saved.")

In [ ]:
top20_final = recommendations.head(20).copy()

top20_final[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "month",
        "risk_score",
        "risk_tier",
        "reason_code",
        "recommended_action",
        "is_declining"
    ]
]

In [ ]:
top20_summary = pd.DataFrame({
    "metric": [
        "Top-20 recommendations",
        "Actual declining items",
        "Precision@20"
    ],
    "value": [
        20,
        int(top20_final["is_declining"].sum()),
        top20_final["is_declining"].mean()
    ]
})

display(top20_summary)

In [ ]:
tier_summary = (
    recommendations
    .groupby("risk_tier", observed=True)
    .size()
    .reset_index(name="count")
)

display(tier_summary)

In [ ]:
top20_final = recommendations.head(20).copy()

top20_final.to_csv(
    "top20_content_recommendations.csv",
    index=False
)

print("top20_content_recommendations.csv saved")

In [ ]:
score_check = pd.DataFrame({
    "metric": [
        "Test rows",
        "Rows with risk score",
        "Rows missing risk score",
        "Risk score coverage"
    ],
    "value": [
        len(recommendations),
        recommendations["risk_score"].notna().sum(),
        recommendations["risk_score"].isna().sum(),
        recommendations["risk_score"].notna().mean()
    ]
})

display(score_check)

In [ ]:

final_recommendations = (
    recommendations[
        recommendations["risk_score"].notna()
    ]
    .sort_values(
        "risk_score",
        ascending=False
    )
    .reset_index(drop=True)
)

final_recommendations["rank"] = (
    np.arange(len(final_recommendations)) + 1
)

print(
    "Rows available for ranking:",
    len(final_recommendations)
)

display(
    final_recommendations.head(20)
)

In [ ]:
final_recommendations[
    FINAL_COLUMNS
].to_csv(
    "ranked_content_recommendations.csv",
    index=False
)

print(
    " Final ranked recommendation CSV saved"
)

print(
    "Rows in final ranking:",
    len(final_recommendations)
)

In [ ]:
final_top20 = final_recommendations.head(20)

final_p20 = (
    final_top20["is_declining"].mean()
)

print("FINAL CAPSTONE METRIC")
print("---------------------")
print(f"Top-20 items: {len(final_top20)}")
print(
    f"Actual declines: "
    f"{int(final_top20['is_declining'].sum())}"
)
print(
    f"Precision@20: "
    f"{final_p20:.3f}"
)

# Final Model Results

The evaluation compared a transparent rule-based baseline with Logistic Regression and Random Forest models using the same chronological test period.

Random Forest produced the strongest overall ranking performance. It achieved a ROC-AUC of 0.5584, an F1 score of 0.5613, and a Precision@20 of 0.550.

For the top-20 ranking objective, 11 of the 20 highest-risk content observations were subsequently classified as declining. This corresponds to a Precision@20 of 55%.

The baseline achieved a Precision@20 of 0.200, while Logistic Regression achieved 0.500. Therefore, the Random Forest ranking captured substantially more future-declining observations among the highest-priority recommendations than the simple rule-based baseline.


In [ ]:
import matplotlib.pyplot as plt

metrics_to_plot = [
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "average_precision",
    "precision_at_20"
]

for metric in metrics_to_plot:

    plt.figure(figsize=(9, 5))

    plt.bar(
        results_df["model"],
        results_df[metric]
    )

    plt.xlabel("Model")
    plt.ylabel(metric.replace("_", " ").title())
    plt.title(
        f"{metric.replace('_', ' ').title()} by Model"
    )

    plt.xticks(rotation=15)
    plt.ylim(0, 1)

    plt.tight_layout()

    filename = f"{metric}_comparison.png"

    plt.savefig(
        filename,
        dpi=200,
        bbox_inches="tight"
    )

    plt.show()

## Top-20 Ranked Recommendations

The Random Forest model was used to rank test observations by predicted probability of future decline.

The top-20 observations represent the highest-priority review queue. Eleven of these twenty observations subsequently experienced the defined decline event, producing a Precision@20 of 0.55.

In [ ]:
top20_plot = final_recommendations.head(20).copy()

plt.figure(figsize=(12, 6))

plt.bar(
    top20_plot["rank"].astype(str),
    top20_plot["risk_score"]
)

plt.xlabel("Recommendation Rank")
plt.ylabel("Predicted Decline Risk")
plt.title("Top-20 Content Decline Risk Ranking")

plt.tight_layout()

plt.savefig(
    "top20_risk_ranking.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()

# Conclusion

This capstone developed a repeatable content-performance decline prioritization workflow using anonymized FlyRank search and content-performance data.

The workflow transformed daily performance observations into monthly content-level features, defined a future decline target, applied leakage checks, and evaluated models using a chronological train/test split.

Random Forest performed best for the primary ranking objective, achieving a Precision@20 of 0.550 compared with 0.500 for Logistic Regression and 0.200 for the rule-based baseline.

The results suggest that current impressions, average position, previous-month position, CTR, and previous-month impressions were among the most informative predictive signals in this modeling setup.

The resulting ranked queue can support content review prioritization. However, the findings are predictive and directional rather than causal. They do not establish how Google's ranking system works or demonstrate that a specific content intervention would cause future performance changes.


# Reproducibility

All analysis in this notebook was performed using the public-safe FlyRank internship warehouse release.

The workflow includes:

1. Data inspection and schema validation
2. Monthly aggregation
3. Feature construction
4. Future decline label construction
5. Leakage checks
6. Chronological train/test splitting
7. Rule-based baseline evaluation
8. Logistic Regression
9. Random Forest
10. Precision@20 evaluation
11. Feature importance analysis
12. Ranked recommendation generation

Random seeds were fixed where applicable to make model training reproducible.